# Curved Multiplier Scaling Backtest

## Question

Does curving **both attack and defence strength multipliers** produce expected
scores that are closer to actual scores?

The existing expected scores are the baseline. For each parameter combination,
the notebook curves:

- the scoring team's attack multiplier;
- the opponent's defence multiplier.

It then recalculates home and away expected scores and compares them with the
actual scores.

```text
error = actual score - expected score
```

A positive mean error means under-prediction. A negative mean error means
over-prediction.


In [1]:
from pathlib import Path
import sqlite3

import numpy as np
import pandas as pd

pd.set_option("display.max_columns", 100)
pd.set_option("display.width", 180)
pd.set_option("display.float_format", lambda value: f"{value:,.4f}")


def find_project_root() -> Path:
    current = Path.cwd().resolve()

    for candidate in [current, *current.parents]:
        if (candidate / "pyproject.toml").exists():
            return candidate

    raise FileNotFoundError(
        "Could not locate the project root containing pyproject.toml."
    )


ROOT = find_project_root()
DB_PATH = ROOT / "data" / "rugby_league_pricing.db"

connection = sqlite3.connect(DB_PATH)

print(f"Project root: {ROOT}")
print(f"Database: {DB_PATH}")


Project root: /workspaces/rugby_league_pricing
Database: /workspaces/rugby_league_pricing/data/rugby_league_pricing.db


## 1. Load completed fixtures, expected scores and strength multipliers


In [2]:
fixtures_query = '''
SELECT
    e.fixture_id,
    f.match_date,
    f.season,
    f.home_team_id,
    f.away_team_id,
    home_team.canonical_name AS home_team,
    away_team.canonical_name AS away_team,
    e.expected_home_score,
    e.expected_away_score,
    r.home_score,
    r.away_score
FROM expected_scores AS e
JOIN fixtures AS f
    ON e.fixture_id = f.fixture_id
JOIN results AS r
    ON e.fixture_id = r.fixture_id
JOIN teams AS home_team
    ON f.home_team_id = home_team.team_id
JOIN teams AS away_team
    ON f.away_team_id = away_team.team_id
WHERE
    r.home_score IS NOT NULL
    AND r.away_score IS NOT NULL
ORDER BY
    f.match_date,
    e.fixture_id
'''

strength_query = '''
SELECT
    fixture_id,
    team_id,
    attack_multiplier,
    defence_multiplier
FROM strength_multipliers
'''

fixtures = pd.read_sql(fixtures_query, connection)
strength = pd.read_sql(strength_query, connection)

fixtures["match_date"] = pd.to_datetime(fixtures["match_date"])

required_strength_columns = {
    "fixture_id",
    "team_id",
    "attack_multiplier",
    "defence_multiplier",
}
missing_columns = required_strength_columns.difference(strength.columns)

if missing_columns:
    raise ValueError(
        f"strength_multipliers is missing columns: {sorted(missing_columns)}"
    )

duplicate_strength_rows = strength.duplicated(
    subset=["fixture_id", "team_id"]
)

if duplicate_strength_rows.any():
    raise ValueError(
        "strength_multipliers contains duplicate fixture/team rows."
    )

print(f"Completed fixtures loaded: {len(fixtures):,}")
print(f"Strength rows loaded: {len(strength):,}")


Completed fixtures loaded: 3,103
Strength rows loaded: 6,208


## 2. Attach home and away attack and defence multipliers


In [26]:
home_strength = strength.rename(
    columns={
        "team_id": "home_team_id",
        "attack_multiplier": "home_attack_multiplier",
        "defence_multiplier": "home_defence_multiplier",
    }
)

away_strength = strength.rename(
    columns={
        "team_id": "away_team_id",
        "attack_multiplier": "away_attack_multiplier",
        "defence_multiplier": "away_defence_multiplier",
    }
)

data = (
    fixtures.merge(
        home_strength,
        on=["fixture_id", "home_team_id"],
        how="left",
        validate="one_to_one",
    )
    .merge(
        away_strength,
        on=["fixture_id", "away_team_id"],
        how="left",
        validate="one_to_one",
    )
)

multiplier_columns = [
    "home_attack_multiplier",
    "home_defence_multiplier",
    "away_attack_multiplier",
    "away_defence_multiplier",
]

missing_multiplier_rows = data[multiplier_columns].isna().any(axis=1)

if missing_multiplier_rows.any():
    missing_count = int(missing_multiplier_rows.sum())
    raise ValueError(
        f"{missing_count} fixtures are missing one or more strength multipliers."
    )

non_positive_rows = data[multiplier_columns].le(0).any(axis=1)

if non_positive_rows.any():
    non_positive_count = int(non_positive_rows.sum())
    raise ValueError(
        f"{non_positive_count} fixtures contain non-positive multipliers."
    )

data.sample(5).T


,2585,2065,307,1387,1811
fixture_id,2023-09-16_4_11,2020-10-09_3_5,2011-05-20_10_5,2016-08-07_10_11,2019-02-24_1_4
match_date,2023-09-16 00:00:00,2020-10-09 00:00:00,2011-05-20 00:00:00,2016-08-07 00:00:00,2019-02-24 00:00:00
season,2023,2020,2011,2016,2019
home_team_id,4,3,10,10,1
away_team_id,11,5,5,11,4
home_team,Hull FC,Catalans Dragons,Salford Red Devils,Salford Red Devils,Wigan Warriors
away_team,Huddersfield Giants,Hull Kingston Rovers,Hull Kingston Rovers,Huddersfield Giants,Hull FC
expected_home_score,27.4615,33.6910,27.1570,33.5323,26.8266
expected_away_score,17.8562,20.1192,24.0582,27.8691,14.5942
home_score,20,34,0,34,22


## 3. Curved scaling


In [27]:
def smooth_scale_multiplier(
    raw_multiplier: pd.Series,
    cap_start: float,
    max_edit: float,
    learning_rate: float = 1.0,
    neutral_multiplier: float = 1.0,
) -> pd.Series:
    """Curve multiplier edits away from the neutral value of 1.0."""
    raw_edit = (
        raw_multiplier - neutral_multiplier
    ) * learning_rate

    curve_position = (
        raw_edit.abs() / cap_start
    ).clip(upper=1.0)

    scaled_magnitude = max_edit * (
        (2.0 * curve_position) - curve_position.pow(2)
    )

    return (
        neutral_multiplier
        + np.sign(raw_edit) * scaled_magnitude
    )


## 4. Recalculate expected scores

The baseline expected scores already contain the original attack and defence
multipliers.

To replace those with curved multipliers without rebuilding every other feature,
the notebook applies multiplier ratios:

```text
curved home expected
=
baseline home expected
× curved home attack / original home attack
× curved away defence / original away defence
```

The away score is recalculated in the same way.


In [28]:
def add_curved_expected_scores(
    frame: pd.DataFrame,
    cap_start: float,
    max_edit: float,
    learning_rate: float,
) -> pd.DataFrame:
    result = frame.copy()

    for column in multiplier_columns:
        result[f"curved_{column}"] = smooth_scale_multiplier(
            raw_multiplier=result[column],
            cap_start=cap_start,
            max_edit=max_edit,
            learning_rate=learning_rate,
        )

    result["curved_expected_home_score"] = (
        result["expected_home_score"]
        * (
            result["curved_home_attack_multiplier"]
            / result["home_attack_multiplier"]
        )
        * (
            result["curved_away_defence_multiplier"]
            / result["away_defence_multiplier"]
        )
    )

    result["curved_expected_away_score"] = (
        result["expected_away_score"]
        * (
            result["curved_away_attack_multiplier"]
            / result["away_attack_multiplier"]
        )
        * (
            result["curved_home_defence_multiplier"]
            / result["home_defence_multiplier"]
        )
    )

    return result


## 5. Actual-versus-expected metrics


In [29]:
def score_metrics(
    actual: pd.Series,
    expected: pd.Series,
    sample: str,
) -> dict[str, float | str]:
    valid = actual.notna() & expected.notna()

    actual_valid = actual.loc[valid].astype(float)
    expected_valid = expected.loc[valid].astype(float)
    error = actual_valid - expected_valid

    mse = np.mean(error.pow(2))

    return {
        "sample": sample,
        "observations": len(error),
        "mean_actual": actual_valid.mean(),
        "mean_expected": expected_valid.mean(),
        "mean_error_actual_minus_expected": error.mean(),
        "absolute_bias": abs(error.mean()),
        "mae": error.abs().mean(),
        "mse": mse,
        "rmse": np.sqrt(mse),
        "correlation": actual_valid.corr(expected_valid),
    }


def evaluate_expected_scores(
    frame: pd.DataFrame,
    home_expected_column: str,
    away_expected_column: str,
) -> pd.DataFrame:
    combined_actual = pd.concat(
        [
            frame["home_score"],
            frame["away_score"],
        ],
        ignore_index=True,
    )

    combined_expected = pd.concat(
        [
            frame[home_expected_column],
            frame[away_expected_column],
        ],
        ignore_index=True,
    )

    return pd.DataFrame(
        [
            score_metrics(
                frame["home_score"],
                frame[home_expected_column],
                "home scores",
            ),
            score_metrics(
                frame["away_score"],
                frame[away_expected_column],
                "away scores",
            ),
            score_metrics(
                combined_actual,
                combined_expected,
                "all team scores",
            ),
        ]
    )


baseline_metrics = evaluate_expected_scores(
    frame=data,
    home_expected_column="expected_home_score",
    away_expected_column="expected_away_score",
)

baseline_metrics


,sample,observations,mean_actual,mean_expected,mean_error_actual_minus_expected,absolute_bias,mae,mse,rmse,correlation
0,home scores,3103,24.6887,25.9480,-1.2593,1.2593,10.5789,180.0920,13.4198,0.3442
1,away scores,3103,20.8975,22.3432,-1.4457,1.4457,9.4003,142.3919,11.9328,0.3809
2,all team scores,6206,22.7931,24.1456,-1.3525,1.3525,9.9896,161.2420,12.6981,0.3805


## 6. Test curved-scaling parameter combinations


In [30]:
CAP_START_VALUES = [
    0.30,
    0.40,
    0.50,
    0.60,
    0.75,
    1.00,
]

MAX_EDIT_VALUES = [
    0.20,
    0.25,
    0.30,
    0.35,
    0.40,
    0.45,
    0.50,
    0.55,
    0.60,
    0.65,
    0.70,
    0.75,
    0.80,
]

LEARNING_RATE_VALUES = [
    0.50,
    0.60,
    0.70,
    0.80,
    0.90,
    1.00,
]

grid_rows = []

for cap_start in CAP_START_VALUES:
    for max_edit in MAX_EDIT_VALUES:
        for learning_rate in LEARNING_RATE_VALUES:
            curved_data = add_curved_expected_scores(
                frame=data,
                cap_start=cap_start,
                max_edit=max_edit,
                learning_rate=learning_rate,
            )

            metrics = evaluate_expected_scores(
                frame=curved_data,
                home_expected_column="curved_expected_home_score",
                away_expected_column="curved_expected_away_score",
            )

            combined_metrics = metrics.loc[
                metrics["sample"] == "all team scores"
            ].iloc[0]

            grid_rows.append(
                {
                    "cap_start": cap_start,
                    "max_edit": max_edit,
                    "learning_rate": learning_rate,
                    **combined_metrics.drop(
                        labels=["sample"]
                    ).to_dict(),
                }
            )

grid_results = (
    pd.DataFrame(grid_rows)
    .sort_values(
        ["mae", "mse", "absolute_bias"],
        ascending=True,
    )
    .reset_index(drop=True)
)

baseline_combined = baseline_metrics.loc[
    baseline_metrics["sample"] == "all team scores"
].iloc[0]

grid_results["mae_change_vs_baseline"] = (
    grid_results["mae"] - baseline_combined["mae"]
)

grid_results["mae_change_percent"] = (
    grid_results["mae_change_vs_baseline"]
    / baseline_combined["mae"]
    * 100.0
)

grid_results["mse_change_vs_baseline"] = (
    grid_results["mse"] - baseline_combined["mse"]
)

grid_results["mse_change_percent"] = (
    grid_results["mse_change_vs_baseline"]
    / baseline_combined["mse"]
    * 100.0
)

grid_results.sample(5).T


,228,145,9,114,417
cap_start,0.7500,0.3000,0.4000,0.4000,0.3000
max_edit,0.5500,0.3000,0.3500,0.2500,0.6000
learning_rate,0.9000,0.7000,0.5000,0.6000,0.7000
observations,"6,206.0000","6,206.0000","6,206.0000","6,206.0000","6,206.0000"
mean_actual,22.7931,22.7931,22.7931,22.7931,22.7931
mean_expected,23.9128,23.6827,23.9122,23.8831,23.4389
mean_error_actual_minus_expected,-1.1197,-0.8896,-1.1191,-1.0900,-0.6458
absolute_bias,1.1197,0.8896,1.1191,1.0900,0.6458
mae,10.0203,9.9175,9.8539,9.8917,11.5446
mse,163.0448,159.2191,155.9432,156.7106,221.2246


## 7. Baseline versus best curved result


In [31]:
best_parameters = grid_results.iloc[0]

best_curved_data = add_curved_expected_scores(
    frame=data,
    cap_start=float(best_parameters["cap_start"]),
    max_edit=float(best_parameters["max_edit"]),
    learning_rate=float(best_parameters["learning_rate"]),
)

best_curved_metrics = evaluate_expected_scores(
    frame=best_curved_data,
    home_expected_column="curved_expected_home_score",
    away_expected_column="curved_expected_away_score",
)

comparison = baseline_metrics.merge(
    best_curved_metrics,
    on="sample",
    suffixes=("_baseline", "_curved"),
    validate="one_to_one",
)

for metric in [
    "absolute_bias",
    "mae",
    "mse",
    "rmse",
]:
    comparison[f"{metric}_change"] = (
        comparison[f"{metric}_curved"]
        - comparison[f"{metric}_baseline"]
    )

    comparison[f"{metric}_change_percent"] = (
        comparison[f"{metric}_change"]
        / comparison[f"{metric}_baseline"]
        * 100.0
    )

comparison[
    [
        "sample",
        "observations_baseline",
        "mean_actual_baseline",
        "mean_expected_baseline",
        "mean_expected_curved",
        "mean_error_actual_minus_expected_baseline",
        "mean_error_actual_minus_expected_curved",
        "mae_baseline",
        "mae_curved",
        "mae_change_percent",
        "mse_baseline",
        "mse_curved",
        "mse_change_percent",
        "rmse_baseline",
        "rmse_curved",
        "correlation_baseline",
        "correlation_curved",
    ]
].T


,0,1,2
sample,home scores,away scores,all team scores
observations_baseline,3103,3103,6206
mean_actual_baseline,24.6887,20.8975,22.7931
mean_expected_baseline,25.9480,22.3432,24.1456
mean_expected_curved,25.7741,22.1098,23.9420
mean_error_actual_minus_expected_baseline,-1.2593,-1.4457,-1.3525
mean_error_actual_minus_expected_curved,-1.0854,-1.2123,-1.1488
mae_baseline,10.5789,9.4003,9.9896
mae_curved,10.4121,9.2934,9.8528
mae_change_percent,-1.5766,-1.1374,-1.3699


In [32]:
print("Best parameters")
print("----------------")
print(
    f"cap_start: {best_parameters['cap_start']:.2f}"
)
print(
    f"max_edit: {best_parameters['max_edit']:.2f}"
)
print(
    f"learning_rate: {best_parameters['learning_rate']:.2f}"
)

combined_comparison = comparison.loc[
    comparison["sample"] == "all team scores"
].iloc[0]

print()
print("Combined actual-versus-expected score result")
print("--------------------------------------------")
print(
    "MAE: "
    f"{combined_comparison['mae_baseline']:.3f} -> "
    f"{combined_comparison['mae_curved']:.3f} "
    f"({combined_comparison['mae_change_percent']:+.2f}%)"
)
print(
    "MSE: "
    f"{combined_comparison['mse_baseline']:.3f} -> "
    f"{combined_comparison['mse_curved']:.3f} "
    f"({combined_comparison['mse_change_percent']:+.2f}%)"
)
print(
    "Mean error: "
    f"{combined_comparison['mean_error_actual_minus_expected_baseline']:.3f} -> "
    f"{combined_comparison['mean_error_actual_minus_expected_curved']:.3f}"
)
print(
    "Correlation: "
    f"{combined_comparison['correlation_baseline']:.3f} -> "
    f"{combined_comparison['correlation_curved']:.3f}"
)


Best parameters
----------------
cap_start: 0.75
max_edit: 0.40
learning_rate: 0.80

Combined actual-versus-expected score result
--------------------------------------------
MAE: 9.990 -> 9.853 (-1.37%)
MSE: 161.242 -> 155.871 (-3.33%)
Mean error: -1.353 -> -1.149
Correlation: 0.380 -> 0.382


## Result interpretation

Use the final comparison table as the decision output.

- Negative MAE/MSE percentage changes are improvements.
- Mean error should remain close to zero.
- Home and away rows should both be checked; an overall improvement should not
  hide a material deterioration on one side.
- Correlation should ideally hold steady or improve.

This notebook tests the attack and defence curves together. It does not add the
curve to production code.


In [11]:
connection.close()
